# Matrix Multiplication on CPU and GPU Using CUDA



## 1. Check GPU and CUDA

In [1]:
!nvidia-smi
!nvcc --version

Tue Aug  4 15:32:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Upload and Extract the Project

In [2]:
from google.colab import files
from pathlib import Path
import zipfile

uploaded = files.upload()
zip_name = next(iter(uploaded))
zip_path = Path('/content') / zip_name

with zipfile.ZipFile(zip_path, 'r') as archive:
    archive.extractall('/content')

project_dir = Path('/content/matrix_cuda_project')
print('Project directory:', project_dir)

Saving matrix_cuda_project_report_fixed.zip to matrix_cuda_project_report_fixed.zip
Project directory: /content/matrix_cuda_project


## 3. Build the Project

In [6]:
%cd /content/matrix_cuda_project_fixed/
!make clean
!make -j2

/content/matrix_cuda_project_fixed
rm -f matrix_bench
rm -rf build
nvcc -O3 -std=c++17 -lineinfo --expt-relaxed-constexpr src/main.cu -o matrix_bench -lcublas -lcusparse
nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
src/main.cu(1107): warning #20208-D: 'long double' is treated as 'double' in device code

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

src/main.cu(1107): warning #20208-D: 'long double' is treated as 'double' in device code

src/main.cu(1115): warning #20208-D: 'long double' is treated as 'double' in device code

src/main.cu(1115): warning #20208-D: 'long double' is treated as 'double' in device code

src/main.cu(1116): warning #20208-D: 'long double' is treated as 'double' in device code

src/main.cu(1117): warning #20208-D: 'long double' is treated as 'double' in device code

src/main.cu(1117): warni

## 4. List CUDA Devices

In [7]:
!./matrix_bench --list-devices

CUDA devices: 1
  [0] Tesla T4 | CC 7.5 | global memory 14.56 GiB | SMs 40


## 5. Quick Test

In [8]:
!mkdir -p results
!./matrix_bench 128 int --iters 2 --warmup 1 --cpu-threads 2 --csv results/quick_results.csv
!./matrix_bench 128 float --iters 2 --warmup 1 --cpu-threads 2 --csv results/quick_results.csv
!./matrix_bench 128 double --iters 2 --warmup 1 --cpu-threads 2 --csv results/quick_results.csv

N=128, type=int, entries per matrix=16384, A sparsity=14.398%
GPU: Tesla T4 (CC 7.5)

Timing summary (mean of measured iterations)
method                         H2D ms    compute ms      D2H ms      total ms     launch us         GOp/s      verify
---------------------------------------------------------------------------------------------------------------------
cpu_naive                      0.0000        2.5474      0.0000        2.5474        0.0000        1.6465        PASS
cpu_blocked_mt                 0.0000        0.8341      0.0000        0.8341        0.0000        5.0288        PASS
cuda_naive                     0.0209        0.0240      0.0108        0.0657        3.6540       63.8286        PASS
cuda_tiled_shared              0.0232        0.0190      0.0148        0.0669        3.3045       62.7139        PASS
cublas_int_via_fp64            0.0306        0.1623      0.0171        0.2206       26.5345       19.0125        PASS
cusparse_int_via_fp64          0.0394      

## 6. Run Benchmarks

In [9]:
!rm -f results/results.csv
!BINARY=./matrix_bench SIZES="64 128 256 512 768 1024" ITERS=5 WARMUP=2 CPU_THREADS=2 bash scripts/run_benchmarks.sh

Running N=64 type=int
N=64, type=int, entries per matrix=4096, A sparsity=14.575%
GPU: Tesla T4 (CC 7.5)

Timing summary (mean of measured iterations)
method                         H2D ms    compute ms      D2H ms      total ms     launch us         GOp/s      verify
---------------------------------------------------------------------------------------------------------------------
cpu_naive                      0.0000        0.2454      0.0000        0.2454        0.0000        2.1367        PASS
cpu_blocked_mt                 0.0000        0.2195      0.0000        0.2195        0.0000        2.3887        PASS
cuda_naive                     0.0140        0.0117      0.0099        0.0451        2.7296       11.6133        PASS
cuda_tiled_shared              0.0133        0.0172      0.0103        0.0505       10.7982       10.3920        PASS
cublas_int_via_fp64            0.0158        0.3118      0.0105        0.3480      184.6910        1.5067        PASS
cusparse_int_via_fp64  

## 7. Run Sparse Matrix Tests

In [10]:
!BINARY=./matrix_bench N=1024 DTYPE=float ITERS=5 WARMUP=2 bash scripts/run_sparse_sweep.sh

N=1024 type=float zero_prob=0.0
N=1024, type=float, entries per matrix=1048576, A sparsity=0.000%
GPU: Tesla T4 (CC 7.5)

Timing summary (mean of measured iterations)
method                         H2D ms    compute ms      D2H ms      total ms     launch us         GOp/s      verify
---------------------------------------------------------------------------------------------------------------------
cuda_tiled_shared              0.6956        5.6515      0.3270        6.6846        4.3452      321.2598        PASS
cublas_sgemm                   0.6950        0.8133      0.3255        1.8442       24.6502     1164.4254        PASS
cusparse_spmm_fp32             1.0425        5.9485      0.3244        7.3256       16.1200      293.1491        PASS

N=1024 type=float zero_prob=0.50
N=1024, type=float, entries per matrix=1048576, A sparsity=49.952%
GPU: Tesla T4 (CC 7.5)

Timing summary (mean of measured iterations)
method                         H2D ms    compute ms      D2H ms      tota

## 8. Analyze Results

In [11]:
!pip install -q -r requirements.txt
!python scripts/analyze_results.py --input results/results.csv --outdir analysis --gpu-name "Tesla T4"
!find analysis -maxdepth 1 -type f | sort


Created figures:
 - compute_throughput_kernel_time_only_tesla_t4_double.png
 - compute_throughput_kernel_time_only_tesla_t4_float.png
 - compute_throughput_kernel_time_only_tesla_t4_int.png
 - gpu_vs_cpu_cross_over_point_total_time_basis_double.png
 - gpu_vs_cpu_cross_over_point_total_time_basis_float.png
 - gpu_vs_cpu_cross_over_point_total_time_basis_int.png
 - time_breakdown_cublas_double.png
 - time_breakdown_cublas_float.png
 - time_breakdown_cublas_int.png
 - time_breakdown_cuda_naive_double.png
 - time_breakdown_cuda_naive_float.png
 - time_breakdown_cuda_naive_int.png
 - time_breakdown_cuda_tiled_double.png
 - time_breakdown_cuda_tiled_float.png
 - time_breakdown_cuda_tiled_int.png
 - time_breakdown_cusparse_double.png
 - time_breakdown_cusparse_float.png
 - time_breakdown_cusparse_int.png
 - total_execution_time_double.png
 - total_execution_time_float.png
 - total_execution_time_int.png

Crossover summary:
data_type  first_tested_N  best_cpu_total_ms  cublas_total_ms  speedup

## 9. Preview Results

In [13]:
import pandas as pd

results = pd.read_csv('/content/matrix_cuda_project_fixed/results/results.csv')
display(results.head(20))

,N,data_type,method,iterations,warmup,cpu_threads,requested_zero_probability,nnz_a,actual_sparsity_a,h2d_ms,...,setup_ms,compute_gops,end_to_end_gops,verified,max_abs_error,max_rel_error,checked_elements,device_name,compute_capability,notes
0,64,int,cpu_naive,5,2,2,0,3499,0.145752,0.000000,...,0.000000,2.136701,2.136701,1,0.0,0.0,4096,Tesla T4,7.5,Native scalar triple-loop CPU implementation.
1,64,int,cpu_blocked_mt,5,2,2,0,3499,0.145752,0.000000,...,0.000000,2.388680,2.388680,1,0.0,0.0,4096,Tesla T4,7.5,Native blocked multi-threaded CPU implementati...
2,64,int,cuda_naive,5,2,2,0,3499,0.145752,0.014042,...,1.069626,44.667394,11.613269,1,0.0,0.0,4096,Tesla T4,7.5,Native CUDA one-thread-per-output-element kernel.
3,64,int,cuda_tiled_shared,5,2,2,0,3499,0.145752,0.013318,...,0.737955,30.544370,10.391983,1,0.0,0.0,4096,Tesla T4,7.5,Native CUDA tiled kernel; 16x16 blocking and s...
4,64,int,cublas_int_via_fp64,5,2,2,0,3499,0.145752,0.015795,...,27.459008,1.681756,1.506658,1,0.0,0.0,4096,Tesla T4,7.5,INT32 emulation through FP64 cuBLAS DGEMM. Exa...
5,64,int,cusparse_int_via_fp64,5,2,2,0,3499,0.145752,0.019424,...,7.658442,23.219955,8.638616,1,0.0,0.0,4096,Tesla T4,7.5,INT32 emulation through FP64 cuSPARSE SpMM; in...
6,128,int,cpu_naive,5,2,2,0,14025,0.143982,0.000000,...,0.000000,1.291284,1.291284,1,0.0,0.0,16384,Tesla T4,7.5,Native scalar triple-loop CPU implementation.
7,128,int,cpu_blocked_mt,5,2,2,0,14025,0.143982,0.000000,...,0.000000,5.332122,5.332122,1,0.0,0.0,16384,Tesla T4,7.5,Native blocked multi-threaded CPU implementati...
8,128,int,cuda_naive,5,2,2,0,14025,0.143982,0.020378,...,1.112480,197.397589,66.197979,1,0.0,0.0,16384,Tesla T4,7.5,Native CUDA one-thread-per-output-element kernel.
9,128,int,cuda_tiled_shared,5,2,2,0,14025,0.143982,0.019066,...,0.766862,237.277334,69.007055,1,0.0,0.0,16384,Tesla T4,7.5,Native CUDA tiled kernel; 16x16 blocking and s...


## 10. Download Results

In [16]:
!cd /content/matrix_cuda_project_fixed && zip -qr /content/matrix_cuda_results.zip results analysis
files.download('/content/matrix_cuda_results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>